In [0]:
# %sql
# DROP TABLE IF EXISTS formula_1.bronze_laps_by_meetings_sessions_drivers;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.formula_1.bronze_laps_by_meetings_sessions_drivers (
  meeting_key STRING,
  meeting_name STRING,
  country_name STRING,
  location STRING,
  date_start STRING,
  date_end STRING,
  session_key STRING,
  session_name STRING,
  session_type STRING,
  circuit_short_name STRING,
  driver_number STRING,
  full_name STRING,
  team_name STRING,
  team_colour STRING,
  lap_number STRING,
  lap_date_start STRING,
  duration_sector_1 STRING,
  duration_sector_2 STRING,
  duration_sector_3 STRING,
  lap_duration STRING,
  i1_speed STRING,
  i2_speed STRING,
  st_speed STRING,
  year INT,
  month INT
)
USING DELTA
PARTITIONED BY (year, meeting_key, session_key, driver_number)

In [0]:
%sql
MERGE INTO workspace.formula_1.bronze_laps_by_meetings_sessions_drivers AS target
USING (
  -- Usamos una CTE para asegurar que cada vuelta sea UNICA
  WITH unique_laps AS (
    SELECT
      meet.meeting_key,
      meet.meeting_name,
      meet.country_name,
      meet.location,
      meet.date_start,
      meet.date_end,
      ses.session_key,
      ses.session_name,
      ses.session_type,
      ses.circuit_short_name,
      driv.driver_number,
      driv.full_name,
      driv.team_name,
      driv.team_colour,
      lap.lap_number,
      lap.date_start AS lap_date_start,
      lap.duration_sector_1,
      lap.duration_sector_2,
      lap.duration_sector_3,
      lap.lap_duration,
      lap.i1_speed,
      lap.i2_speed,
      lap.st_speed,
      CAST(meet.year AS INT) as year,
      CAST(meet.month AS INT) as month,
      -- Creamos un ranking para quedarnos solo con la versión más reciente si hay duplicados
      ROW_NUMBER() OVER (
        PARTITION BY ses.session_key, driv.driver_number, lap.lap_number 
        ORDER BY lap.date_start DESC
      ) as row_num
    FROM formula_1.bronze_meetings meet
    JOIN formula_1.bronze_sessions ses ON meet.meeting_key = ses.meeting_key
    JOIN formula_1.bronze_drivers driv ON meet.meeting_key = driv.meeting_key AND ses.session_key = driv.session_key
    JOIN formula_1.bronze_laps lap ON meet.meeting_key = lap.meeting_key AND ses.session_key = lap.session_key AND driv.driver_number = lap.driver_number
    WHERE meet.meeting_key IS NOT NULL
  )
  SELECT * FROM unique_laps WHERE row_num = 1 -- Solo la fila única
) AS source
ON target.session_key = source.session_key
  AND target.meeting_key = source.meeting_key
  AND target.driver_number = source.driver_number 
  AND target.lap_number = source.lap_number

WHEN MATCHED THEN
  UPDATE SET
    target.meeting_name = source.meeting_name,
    target.team_name = source.team_name,
    target.team_colour = source.team_colour,
    target.lap_duration = source.lap_duration,
    target.st_speed = source.st_speed,
    target.duration_sector_1 = source.duration_sector_1,
    target.duration_sector_2 = source.duration_sector_2,
    target.duration_sector_3 = source.duration_sector_3

WHEN NOT MATCHED THEN
  INSERT *;